# Use Case 1: Ingestion — Scraping Data IDX ke Database

**Kelompok 1 (Extract + Transform + Load)**: Pipeline ETL data ringkasan perdagangan saham dari website Bursa Efek Indonesia (IDX).

---

## Ringkasan Alur (ETL Pipeline)

```
┌─────────┐      ┌───────────┐      ┌──────┐
│ EXTRACT │ ---> │ TRANSFORM │ ---> │ LOAD │
└─────────┘      └───────────┘      └──────┘
  IDX API           Pandas          SQLite
  (JSON)          (DataFrame)      (Database)
```

1. **Extract** — Mengambil data JSON dari API IDX menggunakan `curl_cffi` (bypass Cloudflare TLS fingerprinting). Jika gagal, fallback ke `cloudscraper`.
2. **Transform** — Mengonversi JSON ke DataFrame Pandas, membersihkan data (konversi tipe numerik, hapus baris kosong, standarisasi string).
3. **Load** — Menyimpan data bersih ke database menggunakan SQLAlchemy ORM secara idempotent (tidak duplikat).

## Output

- **Production**: Supabase PostgreSQL (jika env var `SUPABASE_DB_URL` diset)
- **Development**: SQLite (`ingestion/idx_stock.db`) — fallback otomatis

## Mode Operasi

| Environment | Database | Cara Jalan |
|-------------|----------|------------|
| **GitHub Actions** (cron) | Supabase PostgreSQL | Otomatis Senin-Jumat 17:30 WIB |
| **Jupyter lokal / Colab** | SQLite (default) | Run cell by cell |
| **Jupyter + env var** | Supabase PostgreSQL | Set `os.environ['SUPABASE_DB_URL'] = '...'` sebelum run |

---

## Cell [1] — Install Dependensi

### Penjelasan

Cell ini menginstal tiga library utama yang dibutuhkan untuk pipeline ingestion:

| Library | Fungsi |
|---------|--------|
| `curl_cffi` | HTTP client dengan TLS fingerprint impersonation. **Meniru handshake TLS browser Chrome asli** sehingga request kita tidak terdeteksi sebagai bot oleh Cloudflare (proteksi anti-bot website IDX). |
| `cloudscraper` | Library fallback untuk bypass Cloudflare. Jika `curl_cffi` gagal (misal: tidak support OS tertentu), `cloudscraper` akan digunakan sebagai alternatif dengan mekanisme retry. |
| `pandas` | Library manipulasi data tabular. Digunakan untuk membersihkan, mentransformasi, dan menganalisis data hasil scraping dalam bentuk DataFrame. |
| `sqlalchemy` | ORM (Object-Relational Mapping) untuk Python. Memungkinkan kita mendefinisikan tabel database sebagai class Python, sehingga operasi CRUD lebih terstruktur dan mudah di-maintain. |

**Mengapa perlu dua library untuk scraping?** Website IDX dilindungi Cloudflare yang agresif dalam memblokir bot. `curl_cffi` adalah metode utama karena paling akurat meniru browser. `cloudscraper` adalah cadangan jika `curl_cffi` tidak berfungsi di environment tertentu (misalnya Google Colab).

In [1]:
!pip install curl_cffi cloudscraper pandas sqlalchemy

---

## Cell [2] — Import Library

### Penjelasan

Setiap import memiliki peran spesifik dalam pipeline:

| Import | Peran dalam Pipeline |
|--------|---------------------|
| `json` | Parsing dan formatting data JSON dari response API IDX. |
| `time` | Memberi jeda (`sleep`) antar retry request saat fallback ke `cloudscraper`. |
| `os` | Membaca environment variable (`SUPABASE_DB_URL`) untuk koneksi ke Supabase PostgreSQL di production atau GitHub Actions. |
| `curl_cffi.requests as cf_requests` | HTTP GET request dengan impersonate browser Chrome. **Ini kunci bypass Cloudflare**: library ini mengubah TLS fingerprint client agar identik dengan browser asli. |
| `cloudscraper` | Fallback scraper jika `curl_cffi` gagal. Menggunakan engine JavaScript untuk menyelesaikan tantangan Cloudflare. |
| `pandas as pd` | Membangun DataFrame dari data JSON, membersihkan data (drop NA, konversi tipe), dan ekspor CSV. |
| `sqlalchemy.Column, Integer, Float, String` | Tipe data kolom untuk mendefinisikan skema tabel database. |
| `sqlalchemy.create_engine` | Membuat koneksi engine ke database. Support SQLite (local) dan PostgreSQL (Supabase) — otomatis sesuai `DB_URL`. |
| `sqlalchemy.orm.declarative_base` | Base class untuk model ORM — setiap class yang inherit dari `Base` akan menjadi tabel di database. |
| `sqlalchemy.orm.sessionmaker` | Factory untuk membuat session database — session adalah "gerbang" untuk operasi INSERT, UPDATE, DELETE. |

In [2]:
import json
import os
import time
import sys

from curl_cffi import requests as cf_requests
import cloudscraper
import pandas as pd

from sqlalchemy import Column, Integer, Float, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

---

## Cell [3] — Konfigurasi Pipeline

### Penjelasan

Semua parameter yang bisa diubah dikumpulkan di satu tempat agar mudah dikonfigurasi ulang:

| Variabel | Nilai | Keterangan |
|----------|-------|------------|
| `IDX_API_URL` | `https://www.idx.co.id/primary/TradingSummary/GetStockSummary` | Endpoint API IDX yang menyediakan data ringkasan perdagangan saham harian (harga, volume, frekuensi, foreign buy/sell, dll). |
| `IDX_API_PARAMS` | `{"length": 9999, "start": 0}` | Parameter query string. `length=9999` meminta maksimum 9999 record (cukup untuk seluruh saham di IDX, ~900-1000). `start=0` mulai dari record pertama. |
| `DB_URL` | Auto-detect: `SUPABASE_DB_URL` env var → PostgreSQL, jika kosong → SQLite lokal. | **Production (GitHub Actions)**: konek ke Supabase via env var. **Development (local/Colab)**: fallback ke `sqlite:///ingestion/idx_stock.db`. Cukup set env var untuk ganti database — kode ORM tidak perlu diubah. |
| `REQUEST_TIMEOUT` | `30` (detik) | Batas waktu maksimum menunggu response dari server IDX. Jika server tidak merespon dalam 30 detik, request akan dibatalkan dan masuk ke fallback. |

In [3]:
IDX_API_URL = "https://www.idx.co.id/primary/TradingSummary/GetStockSummary"
IDX_API_PARAMS = {"length": 9999, "start": 0}
REQUEST_TIMEOUT = 30

# Database: Supabase (production) atau SQLite (local dev fallback)
SUPABASE_URL = os.getenv("SUPABASE_DB_URL")
if SUPABASE_URL:
    DB_URL = SUPABASE_URL
    DB_MODE = "SUPABASE (PostgreSQL)"
else:
    DB_URL = "sqlite:///idx_stock.db"
    DB_MODE = "SQLITE (lokal)"

print(f"[CONFIG] Database: {DB_MODE}")

[CONFIG] Database: SQLITE (lokal)


---

## Cell [4] — Definisi Model Database (SQLAlchemy ORM)

### Penjelasan Konsep

**ORM (Object-Relational Mapping)** adalah teknik yang memetakan tabel database menjadi class Python. Keuntungan menggunakan ORM dibanding raw SQL:

1. **Type safety** — Setiap kolom memiliki tipe data Python yang jelas (Integer, Float, String).
2. **No SQL injection** — ORM otomatis menggunakan parameterized queries.
3. **Maintainable** — Skema database terdokumentasi langsung di kode sebagai class.
4. **Portable** — Ganti database (SQLite → PostgreSQL) cukup ubah connection string, tanpa ubah kode.

### Penjelasan Per Bagian Kode

```python
Base = declarative_base()
```
Membuat **base class** untuk semua model ORM. Setiap class yang inherit dari `Base` akan otomatis dipetakan ke tabel database. `Base.metadata` menyimpan informasi semua tabel yang terdaftar.

---

```python
class StockSummary(Base):
    __tablename__ = "stock_summary"
```
Mendefinisikan class `StockSummary` yang merepresentasikan **satu baris data** di tabel `stock_summary`. `__tablename__` memberi tahu SQLAlchemy nama tabel di database.

---

**Kolom `id`**:
```python
id = Column(Integer, primary_key=True, autoincrement=True)
```
- **Primary key** — Unique identifier untuk setiap record di database.
- **Autoincrement** — Nilai ID otomatis bertambah (1, 2, 3, ...) setiap insert baru, tidak perlu diisi manual.
- Berbeda dengan `id_stock_summary` (ID dari API IDX), `id` ini adalah ID internal database.

---

**Kolom data saham** (contoh):
```python
stock_code = Column(String, nullable=False, index=True)
```
- `String` — Tipe data teks.
- `nullable=False` — Kolom tidak boleh kosong (NULL). Jika data tidak ada, insert akan gagal.
- `index=True` — Membuat database index pada kolom ini. **Index mempercepat query** `WHERE stock_code = 'BBCA'` karena database tidak perlu scan seluruh tabel.

**Kolom numerik** (contoh):
```python
close = Column(Float, nullable=True)
```
- `Float` — Tipe data angka desimal (cocok untuk harga, volume, value).
- `nullable=True` — Boleh kosong. Beberapa saham mungkin tidak memiliki harga close (misal: saham yang di-suspend).

---

**Method `__repr__`**:
```python
def __repr__(self):
    return f"<StockSummary {self.stock_code} {self.date}>"
```
Method spesial Python yang menentukan bagaimana object ditampilkan saat di-print atau di-debug. Format `<NamaClass atribut>` adalah konvensi standar Python.

---

```python
engine = create_engine(DB_URL, echo=False)
```
- `create_engine` — Membuat koneksi ke database. `engine` adalah objek yang mengelola connection pool dan eksekusi SQL.
- **`DB_URL` otomatis mendeteksi**: jika `SUPABASE_DB_URL` env var diset → konek ke Supabase PostgreSQL; jika tidak → fallback ke SQLite lokal. SQLAlchemy otomatis menyesuaikan SQL dialect berdasarkan connection string.
- `echo=False` — Tidak mencetak log SQL yang dijalankan. Ubah ke `True` untuk debugging.

---

```python
Base.metadata.create_all(engine)
```
- **Membuat semua tabel** yang terdaftar di `Base.metadata` ke database fisik (SQLite, PostgreSQL, atau database apapun yang terhubung).
- Jika tabel sudah ada, operasi ini **tidak melakukan apa-apa** (tidak menghapus data yang sudah ada).
- Equivalent raw SQL: `CREATE TABLE IF NOT EXISTS stock_summary (...)`.  

In [4]:
# ── Base class untuk semua model ORM ──────────────────────────
Base = declarative_base()


# ── Model Tabel stock_summary ─────────────────────────────────
# Setiap instance class ini = 1 baris data ringkasan saham
class StockSummary(Base):
    __tablename__ = "stock_summary"

    # ── Primary key internal database ──
    id = Column(Integer, primary_key=True, autoincrement=True)

    # ── Identitas saham ──
    id_stock_summary = Column(Integer, nullable=False)   # ID unik dari API IDX
    date = Column(String, nullable=False)                # Tanggal perdagangan (YYYY-MM-DD)
    stock_code = Column(String, nullable=False, index=True)  # Kode saham (AALI, BBCA, dll) — di-index untuk query cepat
    stock_name = Column(String, nullable=False)          # Nama emiten lengkap
    remarks = Column(String, nullable=True)              # Keterangan/remarks dari IDX

    # ── Harga (Price) ──
    previous = Column(Float, nullable=True)     # Harga penutupan hari sebelumnya
    open_price = Column(Float, nullable=True)    # Harga pembukaan hari ini
    first_trade = Column(Float, nullable=True)   # Harga transaksi pertama
    high = Column(Float, nullable=True)          # Harga tertinggi hari ini
    low = Column(Float, nullable=True)           # Harga terendah hari ini
    close = Column(Float, nullable=True)         # Harga penutupan hari ini
    change = Column(Float, nullable=True)        # Perubahan harga (close - previous)

    # ── Volume & Nilai Transaksi ──
    volume = Column(Float, nullable=True)        # Volume perdagangan (lot)
    value = Column(Float, nullable=True)         # Nilai transaksi (Rp)
    frequency = Column(Float, nullable=True)     # Frekuensi transaksi
    index_individual = Column(Float, nullable=True)  # Indeks individual

    # ── Bid & Offer ──
    offer = Column(Float, nullable=True)         # Harga penawaran jual terbaik
    offer_volume = Column(Float, nullable=True)   # Volume di harga offer
    bid = Column(Float, nullable=True)            # Harga permintaan beli terbaik
    bid_volume = Column(Float, nullable=True)     # Volume di harga bid

    # ── Saham Beredar ──
    listed_shares = Column(Float, nullable=True)     # Jumlah saham tercatat
    tradeble_shares = Column(Float, nullable=True)   # Jumlah saham dapat diperdagangkan
    weight_for_index = Column(Float, nullable=True)  # Bobot untuk indeks

    # ── Transaksi Asing ──
    foreign_sell = Column(Float, nullable=True)  # Volume penjualan asing
    foreign_buy = Column(Float, nullable=True)   # Volume pembelian asing

    # ── Lain-lain ──
    delisting_date = Column(String, nullable=True)          # Tanggal delisting (jika ada)
    non_regular_volume = Column(Float, nullable=True)      # Volume pasar non-regular
    non_regular_value = Column(Float, nullable=True)        # Nilai pasar non-regular
    non_regular_frequency = Column(Float, nullable=True)   # Frekuensi pasar non-regular

    def __repr__(self):
        """Representasi string untuk debugging: menampilkan kode saham dan tanggal."""
        return f"<StockSummary {self.stock_code} {self.date}>"


# ── Inisialisasi Database ────────────────────────────────────
engine = create_engine(DB_URL, echo=False)
Base.metadata.create_all(engine)
print(f"[DB] Database berhasil dibuat: {DB_URL}")

[DB] Database berhasil dibuat: sqlite:///idx_stock.db


---

## Cell [5] — EXTRACT: Scraping Data dari IDX API

### Konteks: Mengapa Scraping Sulit?

Website IDX (`www.idx.co.id`) dilindungi oleh **Cloudflare**, sebuah layanan keamanan web yang:
1. Mendeteksi apakah pengunjung adalah **browser asli** atau **bot/scraper**.
2. Jika terdeteksi sebagai bot, Cloudflare memblokir request dengan **HTTP 403 (Forbidden)** atau menampilkan halaman **"Checking your browser..."**.

**Solusi**: Dua metode scraping dengan mekanisme fallback:

### Metode 1: `curl_cffi` (Utama — TLS Fingerprint Impersonation)

```python
resp = cf_requests.get(
    IDX_API_URL,
    params=IDX_API_PARAMS,
    impersonate="chrome",      # <-- Kunci: meniru TLS fingerprint Chrome
    timeout=REQUEST_TIMEOUT,
)
```

| Parameter | Fungsi |
|-----------|--------|
| `params=IDX_API_PARAMS` | Otomatis menambahkan `?length=9999&start=0` ke URL. |
| `impersonate="chrome"` | **Ini kunci bypass**. `curl_cffi` mengubah cipher suite, TLS extension, dan HTTP/2 settings client agar **identik dengan Google Chrome asli**. Cloudflare tidak bisa membedakannya. |
| `timeout=30` | Maksimum 30 detik menunggu response. Mencegah program hang. |

### Metode 2: `cloudscraper` (Fallback — JavaScript Challenge Solver)

Jika `curl_cffi` gagal (misalnya: tidak terinstall, error TLS, atau tetap diblokir), program otomatis beralih ke `cloudscraper`:

```python
scraper = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "desktop": True}
)
```

- `cloudscraper` menggunakan **JavaScript engine internal** untuk menyelesaikan tantangan Cloudflare.
- **Retry 3x dengan jeda 3 detik** — Karena Cloudflare kadang butuh beberapa percobaan.

### Response JSON dari IDX API

Struktur response:
```json
{
  "draw": 1,
  "recordsTotal": 959,         // Total saham yang tersedia
  "recordsFiltered": 959,
  "data": [                     // Array of stock summaries
    {
      "IDStockSummary": 4024001,
      "Date": "2026-06-03T00:00:00",
      "StockCode": "AADI",
      "StockName": "Adaro Andalan Indonesia Tbk.",
      "Previous": 8325.0,
      "OpenPrice": 8300.0,
      "High": 8425.0,
      "Low": 7850.0,
      "Close": 8000.0,
      "Volume": 23469400.0,
      ...
    },
    ...
  ]
}
```

In [5]:
print(f"[EXTRACT] Mengambil data dari IDX API ...")
print(f"[EXTRACT] URL: {IDX_API_URL}")
print(f"[EXTRACT] Params: {IDX_API_PARAMS}")
print(f"{'-'*50}")

success = False
response = None
last_error = ""

# ═══════════════════════════════════════════════════════════════
# METODE 0: ScraperAPI — proxy residensial (bypass Cloudflare)
# ═══════════════════════════════════════════════════════════════
# ScraperAPI meneruskan request melalui IP residensial sehingga
# Cloudflare IDX melihatnya sebagai traffic pengguna rumahan,
# bukan bot datacenter. API key diambil dari env var.

SCRAPER_API_KEY = os.getenv("SCRAPER_API_KEY")
if SCRAPER_API_KEY:
    try:
        print("[EXTRACT] Metode 0: ScraperAPI (proxy residensial) ...")
        target_url = f"{IDX_API_URL}?length=9999&start=0"
        import requests as scraper_requests
        resp = scraper_requests.get("http://api.scraperapi.com/", params={
            "api_key": SCRAPER_API_KEY,
            "url": target_url,
        }, timeout=90)
        print(f"[EXTRACT] Status HTTP: {resp.status_code}")
        if resp.status_code == 200:
            payload_check = resp.json()
            if payload_check.get("data"):
                response = resp
                print("[EXTRACT] BERHASIL via ScraperAPI")
                success = True
            else:
                last_error = "ScraperAPI: response kosong"
        else:
            last_error = f"ScraperAPI: HTTP {resp.status_code}"
    except Exception as e:
        last_error = f"ScraperAPI: {e}"
        print(f"[EXTRACT] ScraperAPI gagal: {e}")
else:
    print("[EXTRACT] ScraperAPI dilewati (SCRAPER_API_KEY tidak diset)")

# ═══════════════════════════════════════════════════════════════
# METODE 1: curl_cffi — multi-impersonation
# ═══════════════════════════════════════════════════════════════
# Coba beberapa browser impersonation. IDX kadang menerima
# fingerprint tertentu dan menolak yang lain.

IMPERSONATIONS = ["chrome", "edge", "safari", "firefox"]

for browser in IMPERSONATIONS:
    if success:
        break
    try:
        print(f"[EXTRACT] curl_cffi — impersonate={browser} ...")
        response = cf_requests.get(
            IDX_API_URL,
            params=IDX_API_PARAMS,
            impersonate=browser,
            timeout=REQUEST_TIMEOUT,
        )
        print(f"[EXTRACT] Status HTTP: {response.status_code}")
        if response.status_code == 200:
            print(f"[EXTRACT] BERHASIL via curl_cffi ({browser})")
            success = True
        else:
            last_error = f"HTTP {response.status_code} ({browser})"
    except Exception as e:
        last_error = str(e)
        print(f"[EXTRACT] Gagal ({browser}): {e}")

# ═══════════════════════════════════════════════════════════════
# METODE 2: cloudscraper — dengan retry
# ═══════════════════════════════════════════════════════════════

if not success:
    print(f"\n[EXTRACT] curl_cffi gagal semua. Fallback: cloudscraper ...")

    try:
        scraper = cloudscraper.create_scraper(
            browser={"browser": "chrome", "platform": "windows", "desktop": True}
        )
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Referer": "https://www.idx.co.id/",
            "Accept": "application/json, text/plain, */*",
            "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
            "Cache-Control": "no-cache",
        }

        for attempt in range(1, 5):
            try:
                response = scraper.get(
                    IDX_API_URL,
                    params=IDX_API_PARAMS,
                    timeout=REQUEST_TIMEOUT,
                    headers=headers,
                )
                print(f"[EXTRACT] Retry {attempt}/4 — Status: {response.status_code}")
                if response.status_code == 200:
                    print("[EXTRACT] BERHASIL via cloudscraper")
                    success = True
                    break
            except Exception as retry_err:
                print(f"[EXTRACT] Retry {attempt}/4 — Error: {retry_err}")

            if attempt < 4:
                time.sleep(5)

    except Exception as e:
        last_error = f"cloudscraper: {e}"
        print(f"[EXTRACT] cloudscraper gagal: {e}")

# ═══════════════════════════════════════════════════════════════
# METODE 3: requests biasa — last resort
# ═══════════════════════════════════════════════════════════════

if not success:
    print(f"\n[EXTRACT] Semua metode gagal. Last resort: requests standar ...")
    try:
        import requests
        std_headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Referer": "https://www.idx.co.id/",
            "Accept": "application/json, text/plain, */*",
            "Origin": "https://www.idx.co.id",
        }
        response = requests.get(IDX_API_URL, params=IDX_API_PARAMS, timeout=REQUEST_TIMEOUT, headers=std_headers)
        print(f"[EXTRACT] Status HTTP: {response.status_code}")
        if response.status_code == 200:
            print("[EXTRACT] BERHASIL via requests standar")
            success = True
    except Exception as e:
        last_error = str(e)

if not success:
    msg = f"GAGAL mengambil data setelah semua metode. Error: {last_error}"
    print(f"[EXTRACT] {msg}")
    raise RuntimeError(msg)

# ═══════════════════════════════════════════════════════════════
# Parse JSON response
# ═══════════════════════════════════════════════════════════════

payload = response.json()
records = payload.get("data", [])
total = payload.get("recordsTotal", 0)

print(f"{'-'*50}")
print(f"[EXTRACT] Berhasil mengambil {len(records)} record (total tersedia: {total})")

if len(records) == 0:
    print("[EXTRACT] PERINGATAN: Tidak ada data yang diambil. Periksa API IDX.")
else:
    print(f"\n[EXTRACT] Contoh data pertama (record #1):")
    print(json.dumps(records[0], indent=2, ensure_ascii=False))

[EXTRACT] Mengambil data dari IDX API ...
[EXTRACT] URL: https://www.idx.co.id/primary/TradingSummary/GetStockSummary
[EXTRACT] Params: {'length': 9999, 'start': 0}
--------------------------------------------------
[EXTRACT] ScraperAPI dilewati (SCRAPER_API_KEY tidak diset)
[EXTRACT] curl_cffi — impersonate=chrome ...
[EXTRACT] Status HTTP: 403
[EXTRACT] curl_cffi — impersonate=edge ...
[EXTRACT] Status HTTP: 403
[EXTRACT] curl_cffi — impersonate=safari ...
[EXTRACT] Status HTTP: 200
[EXTRACT] BERHASIL via curl_cffi (safari)
--------------------------------------------------
[EXTRACT] Berhasil mengambil 959 record (total tersedia: 959)

[EXTRACT] Contoh data pertama (record #1):
{
  "No": 1,
  "IDStockSummary": 4025919,
  "Date": "2026-06-05T00:00:00",
  "StockCode": "AADI",
  "StockName": "Adaro Andalan Indonesia Tbk.",
  "Remarks": "--MO1SD0F10000A121------------",
  "Previous": 8025.0,
  "OpenPrice": 7900.0,
  "FirstTrade": 7875.0,
  "High": 7900.0,
  "Low": 7400.0,
  "Close": 7575

---

## Cell [6] — TRANSFORM (Bagian 1): Konversi JSON → DataFrame

### Penjelasan

Data dari API IDX masih dalam bentuk **list of dict** (array of JSON objects). Cell ini mengonversinya menjadi **Pandas DataFrame** — struktur data tabular yang memudahkan pembersihan dan analisis.

**Proses mapping field**:

1. **Iterasi setiap record** dari API response.
2. **Mapping nama field**: API IDX menggunakan PascalCase (`IDStockSummary`, `StockCode`, `OpenPrice`). Kita mapping ke snake_case (Python convention).
3. **`.get()` dengan default value**: Setiap field diakses dengan `item.get("NamaField", default)` untuk menghindari `KeyError` jika field tidak ada.
4. **Truncate date**: `item.get("Date", "")[:10]` — API mengembalikan format `"2026-06-03T00:00:00"`, kita ambil 10 karakter pertama (`"2026-06-03"`).

**Input → Output**:
```
Input:  [{ "IDStockSummary": 4024001, "StockCode": "AADI", ... }, ...]
Output: pd.DataFrame dengan kolom: id_stock_summary, date, stock_code, ...
```

In [6]:
print("[TRANSFORM] Mengonversi JSON → DataFrame ...")

# ── Mapping setiap record JSON ke dictionary Python ──
rows = []
for item in records:
    rows.append({
        # Identitas
        "id_stock_summary": item.get("IDStockSummary"),
        "date": item.get("Date", "")[:10] if item.get("Date") else "",  # "2026-06-03T00:00:00" → "2026-06-03"
        "stock_code": item.get("StockCode", ""),
        "stock_name": item.get("StockName", ""),
        "remarks": item.get("Remarks", ""),

        # Harga
        "previous": item.get("Previous"),
        "open_price": item.get("OpenPrice"),
        "first_trade": item.get("FirstTrade"),
        "high": item.get("High"),
        "low": item.get("Low"),
        "close": item.get("Close"),
        "change": item.get("Change"),

        # Volume & Nilai
        "volume": item.get("Volume"),
        "value": item.get("Value"),
        "frequency": item.get("Frequency"),
        "index_individual": item.get("IndexIndividual"),

        # Bid & Offer
        "offer": item.get("Offer"),
        "offer_volume": item.get("OfferVolume"),
        "bid": item.get("Bid"),
        "bid_volume": item.get("BidVolume"),

        # Saham Beredar
        "listed_shares": item.get("ListedShares"),
        "tradeble_shares": item.get("TradebleShares"),
        "weight_for_index": item.get("WeightForIndex"),

        # Asing
        "foreign_sell": item.get("ForeignSell"),
        "foreign_buy": item.get("ForeignBuy"),
        "delisting_date": item.get("DelistingDate", ""),

        # Non-Regular
        "non_regular_volume": item.get("NonRegularVolume"),
        "non_regular_value": item.get("NonRegularValue"),
        "non_regular_frequency": item.get("NonRegularFrequency"),
    })

# ── Membangun DataFrame dari list of dicts ──
df = pd.DataFrame(rows)

print(f"[TRANSFORM] Data mentah: {len(df)} baris × {len(df.columns)} kolom")
print(f"[TRANSFORM] 5 baris pertama:")
df.head()

[TRANSFORM] Mengonversi JSON → DataFrame ...
[TRANSFORM] Data mentah: 959 baris × 29 kolom
[TRANSFORM] 5 baris pertama:


,id_stock_summary,date,stock_code,stock_name,remarks,previous,open_price,first_trade,high,low,...,bid_volume,listed_shares,tradeble_shares,weight_for_index,foreign_sell,foreign_buy,delisting_date,non_regular_volume,non_regular_value,non_regular_frequency
0,4025919,2026-06-05,AADI,Adaro Andalan Indonesia Tbk.,--MO1SD0F10000A121------------,8025.0,7900.0,7875.0,7900.0,7400.0,...,5000.0,7.786892e+09,7.786892e+09,1.486518e+09,14686700.0,9300900.0,,312.0,2442100.0,4.0
1,4025920,2026-06-05,AALI,Astra Agro Lestari Tbk.,--MO113E100000D232------------,6275.0,6475.0,6475.0,6525.0,6300.0,...,14100.0,1.924688e+09,1.924688e+09,3.907117e+08,3578900.0,1305300.0,,113.0,727425.0,2.0
2,4025921,2026-06-05,ABBA,Mahaka Media Tbk.,--U-4100000000E614-E---------X,34.0,0.0,0.0,33.0,32.0,...,101200.0,3.935893e+09,3.935893e+09,1.298845e+09,0.0,0.0,,0.0,0.0,0.0
3,4025922,2026-06-05,ABDA,Asuransi Bina Dana Arta Tbk.,--UO2105000000G412------------,3000.0,0.0,0.0,3600.0,2550.0,...,100.0,6.208067e+08,6.208067e+08,8.132568e+07,800.0,0.0,,0.0,0.0,0.0
4,4025923,2026-06-05,ABMM,ABM Investama Tbk.,XDMO1135000000A121------------,2300.0,2370.0,2300.0,2370.0,2220.0,...,80600.0,2.753165e+09,2.753165e+09,4.138007e+08,361400.0,95100.0,,65.0,147550.0,1.0


---

## Cell [7] — TRANSFORM (Bagian 2): Pembersihan Data

### Penjelasan

Data mentah dari API seringkali mengandung masalah kualitas. Cell ini melakukan **data cleaning** untuk memastikan data yang masuk ke database sudah bersih dan konsisten.

### Langkah-langkah Pembersihan:

#### 1. Drop baris tanpa stock_code atau date
```python
df = df.dropna(subset=["stock_code", "date"])
```
- `dropna(subset=[...])` — Hapus baris di mana kolom yang disebutkan bernilai `NaN`.
- **Mengapa?** Tanpa `stock_code` dan `date`, record tidak bisa diidentifikasi dan tidak berguna untuk analisis.

#### 2. Konversi kolom numerik
```python
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
```
- `pd.to_numeric(..., errors="coerce")` — Mengonversi nilai ke numerik. Jika gagal (misal: string "N/A"), konversi ke `NaN`.
- **Mengapa `errors="coerce"`?** Lebih aman: data tidak hilang, hanya nilai invalid yang menjadi `NaN` (yang bisa di-handle kemudian).

#### 3. Standarisasi string
```python
df["stock_code"] = df["stock_code"].astype(str).str.strip().str.upper()
```
- `.str.strip()` — Hapus spasi di awal/akhir (misal: `" BBCA "` → `"BBCA"`).
- `.str.upper()` — Ubah ke huruf kapital (konsistensi: `"bbca"` → `"BBCA"`).

### Mengapa Pembersihan Ini Penting?

| Masalah | Tanpa Pembersihan | Setelah Pembersihan |
|---------|-------------------|---------------------|
| Kolom numerik bertipe string | `df['close'].mean()` error | Bisa dihitung statistiknya |
| Kode saham tidak konsisten | `"BBCA"` vs `"bbca"` dianggap berbeda | Semua seragam |
| Spasi di kode saham | Query `WHERE stock_code='BBCA'` gagal | Query berhasil |

In [7]:
print("[TRANSFORM] Membersihkan data ...")

# ── 1. Hapus baris tanpa identitas (stock_code atau date kosong) ──
before_drop = len(df)
df = df.dropna(subset=["stock_code", "date"])
print(f"  - Drop baris tanpa stock_code/date: {before_drop} → {len(df)} (dihapus {before_drop - len(df)})")

# ── 2. Konversi kolom numerik ──
# Semua kolom harga, volume, value, dll harus bertipe float
numeric_cols = [
    "previous", "open_price", "first_trade", "high", "low", "close",
    "change", "volume", "value", "frequency", "index_individual",
    "offer", "offer_volume", "bid", "bid_volume", "listed_shares",
    "tradeble_shares", "weight_for_index", "foreign_sell", "foreign_buy",
    "non_regular_volume", "non_regular_value", "non_regular_frequency",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")  # errors='coerce': invalid → NaN
print(f"  - Konversi {len(numeric_cols)} kolom ke numerik: selesai")

# ── 3. Standarisasi string ──
df["stock_name"] = df["stock_name"].astype(str).str.strip()
df["stock_code"] = df["stock_code"].astype(str).str.strip().str.upper()
print(f"  - String dibersihkan (strip + uppercase)")

print(f"\n[TRANSFORM] Data setelah pembersihan: {len(df)} baris × {len(df.columns)} kolom")
print(f"[TRANSFORM] 10 baris pertama:")
df.head(10)

[TRANSFORM] Membersihkan data ...
  - Drop baris tanpa stock_code/date: 959 → 959 (dihapus 0)
  - Konversi 23 kolom ke numerik: selesai
  - String dibersihkan (strip + uppercase)

[TRANSFORM] Data setelah pembersihan: 959 baris × 29 kolom
[TRANSFORM] 10 baris pertama:


,id_stock_summary,date,stock_code,stock_name,remarks,previous,open_price,first_trade,high,low,...,bid_volume,listed_shares,tradeble_shares,weight_for_index,foreign_sell,foreign_buy,delisting_date,non_regular_volume,non_regular_value,non_regular_frequency
0,4025919,2026-06-05,AADI,Adaro Andalan Indonesia Tbk.,--MO1SD0F10000A121------------,8025.0,7900.0,7875.0,7900.0,7400.0,...,5000.0,7.786892e+09,7.786892e+09,1.486518e+09,14686700.0,9300900.0,,312.0,2442100.0,4.0
1,4025920,2026-06-05,AALI,Astra Agro Lestari Tbk.,--MO113E100000D232------------,6275.0,6475.0,6475.0,6525.0,6300.0,...,14100.0,1.924688e+09,1.924688e+09,3.907117e+08,3578900.0,1305300.0,,113.0,727425.0,2.0
2,4025921,2026-06-05,ABBA,Mahaka Media Tbk.,--U-4100000000E614-E---------X,34.0,0.0,0.0,33.0,32.0,...,101200.0,3.935893e+09,3.935893e+09,1.298845e+09,0.0,0.0,,0.0,0.0,0.0
3,4025922,2026-06-05,ABDA,Asuransi Bina Dana Arta Tbk.,--UO2105000000G412------------,3000.0,0.0,0.0,3600.0,2550.0,...,100.0,6.208067e+08,6.208067e+08,8.132568e+07,800.0,0.0,,0.0,0.0,0.0
4,4025923,2026-06-05,ABMM,ABM Investama Tbk.,XDMO1135000000A121------------,2300.0,2370.0,2300.0,2370.0,2220.0,...,80600.0,2.753165e+09,2.753165e+09,4.138007e+08,361400.0,95100.0,,65.0,147550.0,1.0
5,4025924,2026-06-05,ACES,Aspirasi Hidup Indonesia Tbk.,--MO1835UK6000E743------------,338.0,338.0,338.0,348.0,330.0,...,1827500.0,1.712039e+10,1.712039e+10,6.846444e+09,13354800.0,9140800.0,,0.0,0.0,0.0
6,4025925,2026-06-05,ACRO,Samcro Hyosung Adilestari Tbk.,--UO2100000000E413------------,55.0,54.0,54.0,58.0,52.0,...,83600.0,3.469346e+09,7.492730e+08,7.299503e+08,12000.0,0.0,,0.0,0.0,0.0
7,4025926,2026-06-05,ACST,Acset Indonusa Tbk.,--U-4105000000J211-E---------X,75.0,0.0,0.0,74.0,73.0,...,11100.0,1.767516e+10,1.767516e+10,1.560717e+09,0.0,0.0,,0.0,0.0,0.0
8,4025927,2026-06-05,ADCP,Adhi Commuter Properti Tbk.,--UO2135000000H111M-----------,50.0,50.0,50.0,50.0,50.0,...,0.0,2.222222e+10,2.222222e+10,2.222222e+09,0.0,0.0,,0.0,0.0,0.0
9,4025928,2026-06-05,ADES,Akasha Wira International Tbk.,--UO1135000000D212------------,21450.0,21500.0,21450.0,21975.0,20900.0,...,1700.0,5.898968e+08,5.898968e+08,5.031820e+07,8400.0,600.0,,0.0,0.0,0.0


---

## Cell [8] — LOAD: Menyimpan ke Database SQLite

### Penjelasan

Cell ini menyimpan DataFrame yang sudah bersih ke database SQLite secara **idempotent** (dapat dijalankan berulang kali tanpa menghasilkan duplikasi data).

### Konsep Idempotent Load

**Masalah**: Jika pipeline dijalankan dua kali di hari yang sama, tanpa pengecekan akan terjadi **duplikasi data** (record yang sama tersimpan dua kali).

**Solusi**: Sebelum insert, cek apakah data tanggal tersebut sudah ada:
```
1. Cek: SELECT COUNT(*) FROM stock_summary WHERE date = '2026-06-03'
2. Jika count > 0 → DELETE FROM stock_summary WHERE date = '2026-06-03'
3. INSERT data baru
```

Ini memastikan data untuk tanggal tertentu **selalu fresh** (data terbaru menggantikan data lama).

### Mekanisme Session SQLAlchemy

```python
Session = sessionmaker(bind=engine)  # Factory untuk membuat session
session = Session()                   # Buka session ("koneksi" ke database)

# ... operasi database ...

session.commit()  # Simpan semua perubahan ke database
session.close()   # Tutup session, bebaskan resource
```

**Session** adalah unit of work pattern:
- Semua operasi (insert, update, delete) dikumpulkan dulu di memori.
- `session.commit()` mengeksekusi semua operasi sekaligus (atomic).
- Jika terjadi error sebelum commit, tidak ada yang tersimpan di database (rollback otomatis).

### `bulk_save_objects` vs `add`

```python
session.bulk_save_objects(db_records)  # Bulk insert — jauh lebih cepat

# vs:
# for rec in db_records:
#     session.add(rec)                # Insert satu per satu — lambat
```

- **`bulk_save_objects`**: 1 SQL statement untuk banyak record → **jauh lebih cepat** untuk data besar (~1000 record).
- **`add`**: 1 SQL statement per record → lambat karena overhead per statement.

In [8]:
print("[LOAD] Menyimpan data ke database ...")

# ── Inisialisasi session ──
Session = sessionmaker(bind=engine)
session = Session()

# ── Idempotent: hapus data lama dengan tanggal yang sama ──
# Cek apakah data untuk tanggal ini sudah ada di database
date_val = df["date"].iloc[0] if len(df) > 0 else ""
existing_count = session.query(StockSummary).filter(StockSummary.date == date_val).count()

if existing_count > 0:
    print(f"[LOAD] Data tanggal {date_val} sudah ada ({existing_count} record). Menghapus data lama ...")
    session.query(StockSummary).filter(StockSummary.date == date_val).delete()
    session.commit()
    print(f"[LOAD] Data lama berhasil dihapus.")
else:
    print(f"[LOAD] Data tanggal {date_val} belum ada. Insert baru.")

# ── Konversi DataFrame → List of ORM Objects ──
# Setiap baris DataFrame dikonversi menjadi instance class StockSummary
db_records = []
for _, row in df.iterrows():
    db_records.append(StockSummary(
        id_stock_summary=int(row["id_stock_summary"]) if pd.notna(row["id_stock_summary"]) else 0,
        date=str(row["date"]),
        stock_code=str(row["stock_code"]),
        stock_name=str(row["stock_name"]),
        remarks=str(row.get("remarks", "")),
        previous=float(row["previous"]) if pd.notna(row["previous"]) else None,
        open_price=float(row["open_price"]) if pd.notna(row["open_price"]) else None,
        first_trade=float(row["first_trade"]) if pd.notna(row["first_trade"]) else None,
        high=float(row["high"]) if pd.notna(row["high"]) else None,
        low=float(row["low"]) if pd.notna(row["low"]) else None,
        close=float(row["close"]) if pd.notna(row["close"]) else None,
        change=float(row["change"]) if pd.notna(row["change"]) else None,
        volume=float(row["volume"]) if pd.notna(row["volume"]) else None,
        value=float(row["value"]) if pd.notna(row["value"]) else None,
        frequency=float(row["frequency"]) if pd.notna(row["frequency"]) else None,
        index_individual=float(row["index_individual"]) if pd.notna(row["index_individual"]) else None,
        offer=float(row["offer"]) if pd.notna(row["offer"]) else None,
        offer_volume=float(row["offer_volume"]) if pd.notna(row["offer_volume"]) else None,
        bid=float(row["bid"]) if pd.notna(row["bid"]) else None,
        bid_volume=float(row["bid_volume"]) if pd.notna(row["bid_volume"]) else None,
        listed_shares=float(row["listed_shares"]) if pd.notna(row["listed_shares"]) else None,
        tradeble_shares=float(row["tradeble_shares"]) if pd.notna(row["tradeble_shares"]) else None,
        weight_for_index=float(row["weight_for_index"]) if pd.notna(row["weight_for_index"]) else None,
        foreign_sell=float(row["foreign_sell"]) if pd.notna(row["foreign_sell"]) else None,
        foreign_buy=float(row["foreign_buy"]) if pd.notna(row["foreign_buy"]) else None,
        delisting_date=str(row.get("delisting_date", "")),
        non_regular_volume=float(row["non_regular_volume"]) if pd.notna(row["non_regular_volume"]) else None,
        non_regular_value=float(row["non_regular_value"]) if pd.notna(row["non_regular_value"]) else None,
        non_regular_frequency=float(row["non_regular_frequency"]) if pd.notna(row["non_regular_frequency"]) else None,
    ))

# ── Bulk Insert — semua record dimasukkan dalam 1 operasi ──
session.bulk_save_objects(db_records)
session.commit()

# ── Verifikasi ──
total_count = session.query(StockSummary).count()
print(f"\n[LOAD] {len(db_records)} record berhasil disimpan.")
print(f"[LOAD] Total record di database: {total_count}")
session.close()

[LOAD] Menyimpan data ke database ...
[LOAD] Data tanggal 2026-06-05 sudah ada (959 record). Menghapus data lama ...
[LOAD] Data lama berhasil dihapus.

[LOAD] 959 record berhasil disimpan.
[LOAD] Total record di database: 1918


---

## Cell [9] — Verifikasi: Query Database

### Penjelasan

Cell ini memverifikasi bahwa data benar-benar tersimpan di database. Menggunakan **raw SQL** (via `sqlalchemy.text`) untuk fleksibilitas maksimal dalam query.

```python
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM stock_summary LIMIT 10"))
```

- `engine.connect()` — Membuka koneksi database langsung (tanpa ORM Session). Cocok untuk query read-only.
- `with ... as conn:` — Context manager, otomatis menutup koneksi setelah selesai.
- `text("...")` — Membungkus string SQL mentah agar bisa dieksekusi oleh SQLAlchemy.

### Mengapa Dua Cara Query?

| Metode | Kelebihan | Kekurangan |
|--------|-----------|------------|
| ORM (`session.query(...)`) | Type-safe, autocomplete IDE, portable | Overhead ORM (konversi row → object) |
| Raw SQL (`conn.execute(text(...))`) | Lebih cepat, fleksibel untuk query kompleks | Rentan SQL injection jika tidak hati-hati |

Kita menggunakan **dua metode** untuk menunjukkan keduanya bisa digunakan. Raw SQL untuk verifikasi cepat, ORM untuk operasi data utama.

In [9]:
from sqlalchemy import text

print("[VERIFIKASI] Membaca data dari database ...")

with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM stock_summary LIMIT 10"))
    cols = result.keys()           # Nama kolom dari hasil query
    rows_db = result.fetchall()    # Ambil semua baris hasil query

# Konversi ke DataFrame untuk tampilan yang rapi
df_verify = pd.DataFrame(rows_db, columns=cols)

print(f"[VERIFIKASI] Menampilkan 10 baris pertama dari database:")
df_verify[["stock_code", "stock_name", "close", "volume", "change", "date"]]

[VERIFIKASI] Membaca data dari database ...
[VERIFIKASI] Menampilkan 10 baris pertama dari database:


,stock_code,stock_name,close,volume,change,date
0,AADI,Adaro Andalan Indonesia Tbk.,8000.0,23469400.0,-325.0,2026-06-03
1,AALI,Astra Agro Lestari Tbk.,6650.0,4212500.0,-175.0,2026-06-03
2,ABBA,Mahaka Media Tbk.,37.0,102900.0,0.0,2026-06-03
3,ABDA,Asuransi Bina Dana Arta Tbk.,3120.0,8000.0,-380.0,2026-06-03
4,ABMM,ABM Investama Tbk.,2320.0,1129000.0,-70.0,2026-06-03
5,ACES,Aspirasi Hidup Indonesia Tbk.,348.0,40116300.0,0.0,2026-06-03
6,ACRO,Samcro Hyosung Adilestari Tbk.,60.0,3860700.0,-5.0,2026-06-03
7,ACST,Acset Indonusa Tbk.,79.0,939100.0,-4.0,2026-06-03
8,ADCP,Adhi Commuter Properti Tbk.,50.0,22500.0,0.0,2026-06-03
9,ADES,Akasha Wira International Tbk.,21950.0,100700.0,-275.0,2026-06-03


---

## Cell [10] — Statistik Deskriptif

### Penjelasan

`df.describe()` menghasilkan ringkasan statistik untuk semua kolom numerik:

| Statistik | Arti |
|-----------|------|
| `count` | Jumlah data non-null |
| `mean` | Rata-rata |
| `std` | Standar deviasi (sebaran data) |
| `min` | Nilai minimum |
| `25%` | Kuartil pertama (25% data di bawah nilai ini) |
| `50%` | Median (50% data di bawah nilai ini) |
| `75%` | Kuartil ketiga (75% data di bawah nilai ini) |
| `max` | Nilai maksimum |

**Gunakan statistik ini untuk**:
- Mendeteksi outlier (max/min yang tidak wajar)
- Memahami distribusi data (misal: rata-rata volume vs median — jika beda jauh, ada outlier)
- Validasi hasil scraping (misal: count harus ~900-1000 untuk IDX)

In [10]:
print("[STATISTIK] Ringkasan statistik data:")
df.describe()

[STATISTIK] Ringkasan statistik data:


,id_stock_summary,previous,open_price,first_trade,high,low,close,change,volume,value,...,bid,bid_volume,listed_shares,tradeble_shares,weight_for_index,foreign_sell,foreign_buy,non_regular_volume,non_regular_value,non_regular_frequency
count,9.590000e+02,959.000000,959.000000,959.000000,959.000000,959.000000,959.000000,959.000000,9.590000e+02,9.590000e+02,...,959.000000,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,959.00000
mean,4.026398e+06,1222.787278,588.454640,588.297185,1094.892596,1021.245047,1186.202294,-36.584984,2.527786e+07,2.243874e+10,...,1031.028154,6.506610e+05,1.308886e+10,1.287635e+10,4.100741e+09,8.760123e+06,7.383478e+06,1.161280e+07,1.062724e+10,0.68926
std,2.769838e+02,6742.656140,1816.897388,1816.351607,6516.705477,6157.372565,6426.598007,350.130967,1.694349e+08,2.137185e+11,...,6194.038211,5.689365e+06,4.582396e+10,4.569681e+10,2.951355e+10,7.355173e+07,5.247572e+07,2.962209e+08,2.844990e+11,5.92678
min,4.025919e+06,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-10400.000000,0.000000e+00,0.000000e+00,...,0.000000,0.000000e+00,3.600000e+06,3.600000e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00000
25%,4.026158e+06,88.000000,0.000000,0.000000,73.000000,64.500000,82.500000,-18.000000,6.400000e+04,2.575330e+07,...,54.000000,1.000000e+02,1.460281e+09,1.388310e+09,2.353932e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00000
50%,4.026398e+06,226.000000,88.000000,89.000000,194.000000,180.000000,214.000000,-5.000000,1.116900e+06,2.307855e+08,...,170.000000,8.700000e+03,3.571452e+09,3.534000e+09,7.245845e+08,2.500000e+04,2.000000e+04,0.000000e+00,0.000000e+00,0.00000
75%,4.026638e+06,747.500000,348.000000,349.000000,712.500000,662.500000,715.000000,0.000000,9.398900e+06,1.957115e+09,...,640.000000,1.219000e+05,1.033562e+10,1.023714e+10,2.427630e+09,1.296900e+06,1.061950e+06,0.000000e+00,0.000000e+00,0.00000
max,4.026877e+06,190650.000000,22025.000000,22000.000000,190625.000000,180100.000000,180250.000000,1000.000000,3.789100e+09,5.486487e+12,...,180275.000000,1.415533e+08,1.140573e+12,1.140573e+12,8.774430e+11,1.884958e+09,1.046870e+09,9.147693e+09,8.802133e+12,161.00000


---

## Cell [11] — Ekspor ke CSV (Opsional)

### Penjelasan

Mengekspor DataFrame ke file CSV sebagai backup atau untuk digunakan oleh notebook modelling. Format CSV memudahkan:
- Dibuka di Excel / Google Sheets
- Di-load oleh notebook lain tanpa perlu koneksi database
- Di-share ke anggota tim

```python
df.to_csv(csv_path, index=False)
```
- `index=False` — Jangan sertakan index Pandas (0, 1, 2, ...) sebagai kolom di CSV.

### Ringkasan Pipeline

Di akhir cell ini, pipeline ETL selesai:
```
IDX API (JSON) → Pandas DataFrame → SQLite Database + CSV File
     ✓                 ✓                  ✓
  Extract           Transform            Load
```

In [11]:
csv_path = "idx_stock_summary.csv"
df.to_csv(csv_path, index=False)

print(f"[EXPORT] Data diekspor ke: {csv_path}")
print(f"[EXPORT] Total: {len(df)} baris × {len(df.columns)} kolom")
print(f"\n{'='*60}")
print(f"  PIPELINE SELESAI!")
print(f"  - Database : {DB_MODE}")
print(f"  - CSV      : {csv_path}")
print(f"  - Record   : {len(df)} saham")
print(f"{'='*60}")

[EXPORT] Data diekspor ke: idx_stock_summary.csv
[EXPORT] Total: 959 baris × 29 kolom

  PIPELINE SELESAI!
  - Database : SQLITE (lokal)
  - CSV      : idx_stock_summary.csv
  - Record   : 959 saham
